# LIANA: a LIgand-receptor ANalysis frAmework


In [ ]:
# use env liana_r

if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

if (!requireNamespace("remotes", quietly = TRUE))
    install.packages("remotes")

if (!requireNamespace("tidyverse", quietly = TRUE))
    install.packages("tidyverse")

if (!requireNamespace("magrittr", quietly = TRUE))
    install.packages("magrittr")

if (!requireNamespace("scran", quietly = TRUE)) {
    install.packages("scran")
}

In [ ]:
library(tidyverse)
library(magrittr)
library(liana)
library(Seurat)
library(stringr)


In [ ]:
show_resources()

show_methods()

In [ ]:
data_path <- "/nfs/home/students/i.kaciran/FoPra_PLAs/data/datasets/"
output_dir <- "../../results/exploration_week1/"

# "gated_ImmuneAging.rds", "gated_heart_processed.rds", "gated_sepsis_processed.rds", "gated_vaccine_processed.rds", "gated_our_dataset_processed.rds"
file_name <- "gated_ImmuneAging.rds"
#file_name <- "gated_heart_processed.rds" 
#file_name <- "gated_sepsis_processed.rds" 
#file_name <- "gated_vaccine_processed.rds" 
#file_name <- "gated_our_dataset_processed.rds" 

if (file_name == "gated_ImmuneAging.rds"){
    dataset_type <- "immune_aging"
} else if (file_name == "gated_heart_processed.rds"){
    dataset_type <- "heart"
} else if (file_name == "gated_sepsis_processed.rds"){
    dataset_type <- "sepsis"
} else if (file_name == "gated_vaccine_processed.rds"){
    dataset_type <- "vaccine"
} else if (file_name == "gated_our_dataset_processed.rds"){
    dataset_type <- "our_data"
}


In [ ]:
# Load data
seurat_obj <- readRDS(paste0(data_path, file_name))

In [ ]:
run_and_save_liana <- function(seurat_obj, dataset_name, output_dir = "results/liana") {
    dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
    message("Running LIANA for: ", dataset_name)

    liana_raw <- liana_wrap(seurat_obj)

    liana_consensus <- liana_raw %>%
    liana_aggregate() %>%
    arrange(aggregate_rank)

    # Count cells per cell type
    cell_type_counts_table <- table(Idents(seurat_obj))

    cell_type_counts <- tibble(
    cell_type = names(cell_type_counts_table),
    n_cells = as.integer(cell_type_counts_table)
    ) %>%
    arrange(desc(n_cells))

    # Collect everything that should be saved
    liana_result <- list(
    metadata = list(
        dataset_name = dataset_name,
        created_at = Sys.time(),
        n_cells = ncol(seurat_obj),
        n_features = nrow(seurat_obj),
        default_assay = DefaultAssay(seurat_obj),
        methods = names(liana_raw),
        R_version = R.version.string,
        liana_version = as.character(packageVersion("liana")),
        Seurat_version = as.character(packageVersion("Seurat"))
    ),

    cell_type_counts = cell_type_counts,

    raw = liana_raw,

    consensus = liana_consensus
)

  output_file <- file.path(
    output_dir,
    paste0(dataset_name, "_liana_results.rds")
  )

  saveRDS(
    liana_result,
    file = output_file,
    compress = "xz"
  )

  message("Saved LIANA results to: ", output_file)

  invisible(liana_result)
}

In [ ]:
liana_result <- run_and_save_liana(
  seurat_obj = seurat_obj,
  dataset_name = "my_dataset"
)

In [ ]:
analyze_liana <- function(
    liana_result,
    top_n = 50,
    aggregate_rank_threshold = 0.01,
    genes_of_interest = NULL,
    source_groups = NULL,
    target_groups = NULL
) {
  consensus <- liana_result$consensus %>%
    arrange(aggregate_rank)

  # Optional source-cell filter
  if (!is.null(source_groups)) {
    consensus <- consensus %>%
      filter(source %in% source_groups)
  }

  # Optional target-cell filter
  if (!is.null(target_groups)) {
    consensus <- consensus %>%
      filter(target %in% target_groups)
  }

  top_interactions <- consensus %>%
    slice_head(n = top_n)

  significant_interactions <- consensus %>%
    filter(
      !is.na(aggregate_rank),
      aggregate_rank <= aggregate_rank_threshold
    )

  source_target_summary <- significant_interactions %>%
    group_by(source, target) %>%
    summarise(
      n_interactions = n(),
      n_unique_lr_pairs = n_distinct(
        ligand.complex,
        receptor.complex
      ),
      best_aggregate_rank = min(
        aggregate_rank,
        na.rm = TRUE
      ),
      median_aggregate_rank = median(
        aggregate_rank,
        na.rm = TRUE
      ),
      .groups = "drop"
    ) %>%
    arrange(
      desc(n_interactions),
      best_aggregate_rank
    )

  ligand_summary <- significant_interactions %>%
    group_by(ligand.complex) %>%
    summarise(
      n_interactions = n(),
      n_source_types = n_distinct(source),
      n_target_types = n_distinct(target),
      best_aggregate_rank = min(
        aggregate_rank,
        na.rm = TRUE
      ),
      .groups = "drop"
    ) %>%
    arrange(
      desc(n_interactions),
      best_aggregate_rank
    )

  receptor_summary <- significant_interactions %>%
    group_by(receptor.complex) %>%
    summarise(
      n_interactions = n(),
      n_source_types = n_distinct(source),
      n_target_types = n_distinct(target),
      best_aggregate_rank = min(
        aggregate_rank,
        na.rm = TRUE
      ),
      .groups = "drop"
    ) %>%
    arrange(
      desc(n_interactions),
      best_aggregate_rank
    )

  lr_pair_summary <- significant_interactions %>%
    group_by(
      ligand.complex,
      receptor.complex
    ) %>%
    summarise(
      n_interactions = n(),
      n_source_target_pairs = n_distinct(
        paste(source, target, sep = " -> ")
      ),
      best_aggregate_rank = min(
        aggregate_rank,
        na.rm = TRUE
      ),
      .groups = "drop"
    ) %>%
    arrange(
      desc(n_source_target_pairs),
      best_aggregate_rank
    )

  gene_interactions <- NULL
  gene_significant <- NULL

  if (!is.null(genes_of_interest)) {
    gene_pattern <- paste0(
      "(^|[^[:alnum:]])(",
      paste(genes_of_interest, collapse = "|"),
      ")([^[:alnum:]]|$)"
    )

    gene_interactions <- consensus %>%
      filter(
        str_detect(
          coalesce(as.character(ligand.complex), ""),
          regex(gene_pattern, ignore_case = FALSE)
        ) |
          str_detect(
            coalesce(as.character(receptor.complex), ""),
            regex(gene_pattern, ignore_case = FALSE)
          )
      )

    gene_significant <- gene_interactions %>%
      filter(
        !is.na(aggregate_rank),
        aggregate_rank <= aggregate_rank_threshold
      )
  }

  list(
    parameters = list(
      top_n = top_n,
      aggregate_rank_threshold = aggregate_rank_threshold,
      genes_of_interest = genes_of_interest,
      source_groups = source_groups,
      target_groups = target_groups
    ),

    filtered_consensus = consensus,
    top_interactions = top_interactions,
    significant_interactions = significant_interactions,
    source_target_summary = source_target_summary,
    ligand_summary = ligand_summary,
    receptor_summary = receptor_summary,
    lr_pair_summary = lr_pair_summary,
    gene_interactions = gene_interactions,
    gene_significant = gene_significant
  )
}

In [ ]:
liana_result <- readRDS(
  "results/liana/my_dataset_liana_results.rds"
)

overview <- analyze_liana(
  liana_result = liana_result,
  top_n = 30,
  aggregate_rank_threshold = 0.01
)

overview_top100 <- analyze_liana(
  liana_result = liana_result,
  top_n = 100,
  aggregate_rank_threshold = 0.05
)

lymphocyte_overview <- analyze_liana(
  liana_result = liana_result,
  top_n = 50,
  aggregate_rank_threshold = 0.01,
  source_groups = c("B", "NK", "CD8 T"),
  target_groups = c("B", "NK", "CD8 T")
)

In [ ]:
pla_signaling_genes <- list(

  adhesion = c(
    "SELP",
    "SELPLG",
    "ITGAM",
    "ITGB2",
    "GP1BA",
    "GP1BB",
    "GP9",
    "GP5",
    "VWF",
    "ITGA2B",
    "ITGB3",
    "ICAM2",
    "ITGAL",
    "CD40LG",
    "CD40",
    "PDPN",
    "CLEC1B",
    "FCGR2A"
  ),

  cytokines_chemokines = c(
    "CXCL8",
    "CXCR1",
    "CXCR2",
    "CCL2",
    "CCR2",
    "PF4",
    "CCL5",
    "CCR1",
    "CCR3",
    "CCR5",
    "IL1B",
    "IL1R1",
    "IL1RAP"
  ),

  lipid_mediator_pathways = c(
    "PTGS2",
    "TBXAS1",
    "TBXA2R",
    "PTAFR"
  )
)

pla_genes <- unique(unlist(pla_signaling_genes))

pla_liana_genes <- unique(c(
  pla_signaling_genes$adhesion,
  pla_signaling_genes$cytokines_chemokines
))

pla_context_genes <- pla_signaling_genes$lipid_mediator_pathways

pla_overview <- analyze_liana(
  liana_result = liana_result,
  top_n = 50,
  aggregate_rank_threshold = 0.01,
  genes_of_interest = pla_liana_genes
)

In [ ]:
overview$top_interactions
overview$significant_interactions
overview$source_target_summary
overview$ligand_summary
overview$receptor_summary
overview$lr_pair_summary

pla_overview$gene_significant

In [ ]:
liana_result$consensus %>%
  liana_dotplot(
    source_groups = "B",
    target_groups = c("NK", "CD8 T", "B"),
    ntop = 20
  )

In [ ]:
heat_freq(
  overview$significant_interactions
)

In [ ]:
chord_freq(
  overview$significant_interactions,
  source_groups = c("CD8 T", "NK", "B"),
  target_groups = c("CD8 T", "NK", "B")
)

In [ ]:
saveRDS(
  pla_overview,
  "results/liana/my_dataset_pla_overview.rds"
)

## Tutorial

In [ ]:
liana_path <- system.file(package = "liana")
testdata <-
  readRDS(file.path(liana_path , "testdata", "input", "testdata.rds"))

testdata %>% dplyr::glimpse()

In [ ]:
# Run liana
liana_test <- liana_wrap(testdata)

# Liana returns a list of results, each element of which corresponds to a method
liana_test %>% dplyr::glimpse()

In [ ]:
# top natmi results
names(liana_test)
unique(liana_test$natmi$source)
unique(liana_test$natmi$target)

natmi_res <- liana_test$natmi

top_natmi <- natmi_res %>%
  arrange(desc(prod_weight)) %>%
  select(
    source,
    target,
    ligand.complex,
    receptor.complex,
    ligand.expr,
    receptor.expr,
    ligand.prop,
    receptor.prop,
    prod_weight,
    edge_specificity
  ) %>%
  head(30)

top_natmi

In [ ]:
pla_signaling_genes <- list(

  adhesion = c(
    "SELP",
    "SELPLG",
    "ITGAM",
    "ITGB2",
    "GP1BA",
    "GP1BB",
    "GP9",
    "GP5",
    "VWF",
    "ITGA2B",
    "ITGB3",
    "ICAM2",
    "ITGAL",
    "CD40LG",
    "CD40",
    "PDPN",
    "CLEC1B",
    "FCGR2A"
  ),

  cytokines_chemokines = c(
    "CXCL8",
    "CXCR1",
    "CXCR2",
    "CCL2",
    "CCR2",
    "PF4",
    "CCL5",
    "CCR1",
    "CCR3",
    "CCR5",
    "IL1B",
    "IL1R1",
    "IL1RAP"
  ),

  lipid_mediator_pathways = c(
    "PTGS2",
    "TBXAS1",
    "TBXA2R",
    "PTAFR"
  )
)

pla_genes <- unique(unlist(pla_signaling_genes))

natmi_res %>%
  filter(
    ligand %in% pla_genes |
      receptor %in% pla_genes |
      ligand.complex %in% pla_genes |
      receptor.complex %in% pla_genes
  ) %>%
  arrange(desc(prod_weight))

In [ ]:
lapply(names(liana_test), function(method) {
  res <- liana_test[[method]]
  message("\n--- ", method, " ---")
  print(head(res))
})

In [ ]:
tibble(
  method = names(liana_test),
  n_interactions = sapply(liana_test, nrow)
)

In [ ]:
combined <- bind_rows(
  liana_test,
  .id = "method"
)

combined %>%
  select(method, source, target, ligand.complex, receptor.complex, everything())
  
common_lr <- combined %>%
  count(source, target, ligand.complex, receptor.complex, sort = TRUE)

common_lr %>%
  filter(n >= 3) %>%
  head(30)

In [ ]:
# cellphone db results
cpdb_res <- liana_test$cellphonedb

cpdb_res %>%
  arrange(pvalue) %>%
  head(30)

In [ ]:
# We can aggregate these results into a tibble with consensus ranks
liana_test <- liana_test %>%
  liana_aggregate()

dplyr::glimpse(liana_test)

### Dotplot

In [ ]:
liana_test %>%
  liana_dotplot(source_groups = c("B"),
                target_groups = c("NK", "CD8 T", "B"),
                ntop = 20)

### Frequency Heatmap

In [ ]:
liana_trunc <- liana_test %>%
   # only keep interactions concordant between methods
  filter(aggregate_rank <= 0.01) # note that these pvals are already corrected

heat_freq(liana_trunc)

### Frequency Chord diagram

In [ ]:
if(!require("circlize")){
  install.packages("circlize", quiet = TRUE,
                   repos = "http://cran.us.r-project.org")
}

In [ ]:
p <- chord_freq(liana_trunc,
                source_groups = c("CD8 T", "NK", "B"),
                target_groups = c("CD8 T", "NK", "B"))

### Running a method of choice

In [ ]:
# Load Sce testdata
sce <- readRDS(file.path(liana_path , "testdata", "input", "testsce.rds"))

# RUN CPDB alone
cpdb_test <- liana_wrap(sce,
                        method = 'cellphonedb',
                        resource = c('CellPhoneDB'),
                        permutation.params = list(nperms=100,
                                                  parallelize=FALSE,
                                                  workers=4),
                        expr_prop=0.05)

# identify interactions of interest
cpdb_int <- cpdb_test %>%
  # only keep interactions with p-val <= 0.05
  filter(pvalue <= 0.05) %>% # this reflects interactions `specificity`
  # then rank according to `magnitude` (lr_mean in this case)
  rank_method(method_name = "cellphonedb",
              mode = "magnitude") %>%
  # keep top 20 interactions (regardless of cell type)
  distinct_at(c("ligand.complex", "receptor.complex")) %>%
  head(20)



# Plot toy results
cpdb_test %>%
  # keep only the interactions of interest
  inner_join(cpdb_int, 
             by = c("ligand.complex", "receptor.complex")) %>%
  # invert size (low p-value/high specificity = larger dot size)
  # + add a small value to avoid Infinity for 0s
  mutate(pvalue = -log10(pvalue + 1e-10)) %>% 
  liana_dotplot(source_groups = c("c"),
                target_groups = c("c", "a", "b"),
                specificity = "pvalue",
                magnitude = "lr.mean",
                show_complex = TRUE,
                size.label = "-log10(p-value)")

In [ ]:
#remotes::install_github("jinworks/CellChat")